In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data_claim = pd.read_csv('data/Data_Klaim.csv')
data_polis = pd.read_csv('data/Data_Polis.csv')

In [3]:
data_claim.head()

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS
0,C-0001-M,POL-0176,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,2024-07-08,2024-05-27,2024-05-27,28093653.0,6.143948e+06,Singapore
1,C-0002-M,POL-3288,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-08-06,2024-07-15,2024-07-15,80987278.0,8.230952e+07,Malaysia
2,C-0003-M,POL-1786,R,OP,C18.9,"MALIGNANT NEOPLASM, COLON, UNSPECIFIED",PAID,2024-10-17,2024-05-16,2024-05-16,183047130.0,1.928599e+08,Singapore
3,C-0004-M,POL-1786,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-09-03,2024-07-18,2024-07-18,191424386.0,1.914244e+08,Singapore
4,C-0005-M,POL-2778,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,NaN,2024-06-06,2024-06-06,138936357.0,1.389364e+08,Singapore


In [4]:
data_polis.head()

,Nomor Polis,Plan Code,Gender,Tanggal Lahir,Tanggal Efektif Polis,Domisili
0,POL-0001,M-003,M,19640811,20140603,JAKARTA
1,POL-0002,M-003,M,19710730,20140603,JAKARTA
2,POL-0003,M-001,M,19790821,20160808,JAKARTA
3,POL-0004,M-003,M,20140724,20160811,JAKARTA
4,POL-0005,M-001,F,19810114,20150828,JAKARTA


Actuarial LightGBM

In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# STEP 1: DATA PREPARATION & MERGING
# ==========================================
print("1. Processing Raw Data...")

# 1. Convert Dates
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
data_polis['Tanggal Lahir'] = pd.to_datetime(data_polis['Tanggal Lahir'], format='%Y%m%d', errors='coerce')

# 2. Merge Data (Optional: use polis data for demographics if needed, but keeping it focused on claims)
merged_df = pd.merge(data_claim, data_polis, on='Nomor Polis', how='left')

# 3. Extract Year and Month from Admission Date (Tanggal Masuk)
merged_df['Year'] = merged_df['Tanggal Pasien Masuk RS'].dt.year
merged_df['Month'] = merged_df['Tanggal Pasien Masuk RS'].dt.month
merged_df['Period'] = merged_df['Tanggal Pasien Masuk RS'].dt.to_period('M')

# 4. Filter only PAID claims (Safety check)
valid_claims = merged_df[merged_df['Status Klaim'] == 'PAID'].copy()

# ==========================================
# STEP 2: MONTHLY AGGREGATION
# ==========================================
print("2. Aggregating Monthly Data...")

monthly_data = []
for period, group in valid_claims.groupby('Period'):
    monthly_data.append({
        'Period': period,
        'Year': period.year,
        'Month': period.month,
        'Frequency': len(group), # Raw Count
        'Severity': group['Nominal Klaim Yang Disetujui'].mean(),
        'Total_Claim': group['Nominal Klaim Yang Disetujui'].sum()
    })

monthly_df = pd.DataFrame(monthly_data).sort_values('Period').reset_index(drop=True)

# ==========================================
# STEP 3: FEATURE ENGINEERING
# ==========================================
print("3. Engineering Time-Series Features...")

# Seasonality
monthly_df['month_sin'] = np.sin(2 * np.pi * monthly_df['Month']/12)
monthly_df['month_cos'] = np.cos(2 * np.pi * monthly_df['Month']/12)

# Lags & Rolling Means
targets = ['Frequency', 'Severity', 'Total_Claim']
for col in targets:
    monthly_df[f'{col}_Lag1'] = monthly_df[col].shift(1)
    monthly_df[f'{col}_Lag2'] = monthly_df[col].shift(2)
    monthly_df[f'{col}_Lag3'] = monthly_df[col].shift(3)
    monthly_df[f'{col}_RollMean3'] = monthly_df[col].rolling(window=3).mean()

# Drop initial NaN rows
train_df = monthly_df.dropna().copy()

# Define feature sets
features_freq = ['month_sin', 'month_cos', 'Frequency_Lag1', 'Frequency_Lag2', 'Frequency_Lag3', 'Frequency_RollMean3']
features_sev = ['month_sin', 'month_cos', 'Severity_Lag1', 'Severity_Lag2', 'Severity_Lag3', 'Severity_RollMean3']
features_total = ['month_sin', 'month_cos', 'Total_Claim_Lag1', 'Total_Claim_Lag2', 'Total_Claim_Lag3', 'Total_Claim_RollMean3']

# ==========================================
# STEP 4: TRAIN ACTUARIAL MODELS
# ==========================================
print("4. Training Actuarial LightGBM Models...")

# Base parameters optimized for small datasets
base_params = {
    'boosting_type': 'gbdt',
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 15,          # Kept very small to prevent overfitting on small data
    'min_child_samples': 3,
    'subsample': 0.8,
    'random_state': 42,
    'verbose': -1
}

# Model 1: Frequency (POISSON) - Predicts exact raw numbers naturally
params_freq = base_params.copy()
params_freq['objective'] = 'poisson'
model_freq = lgb.LGBMRegressor(**params_freq)
model_freq.fit(train_df[features_freq], train_df['Frequency'])

# Model 2: Severity (GAMMA) - Handles skewed positive costs
params_sev = base_params.copy()
params_sev['objective'] = 'gamma'
model_sev = lgb.LGBMRegressor(**params_sev)
model_sev.fit(train_df[features_sev], train_df['Severity'])

# Model 3: Total Claim (TWEEDIE) - The ultimate insurance aggregate model
params_total = base_params.copy()
params_total['objective'] = 'tweedie'
params_total['tweedie_variance_power'] = 1.5 # Standard for Compound Poisson-Gamma
model_total = lgb.LGBMRegressor(**params_total)
model_total.fit(train_df[features_total], train_df['Total_Claim'])

# ==========================================
# STEP 5: RECURSIVE FORECASTING
# ==========================================
print("5. Generating Forecast (Aug - Dec 2025)...")

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=monthly_df['Period'].max() + 1, end=target_end, freq='M')

current_history = monthly_df.copy()
predictions = []

for period in forecast_range:
    p_month = period.month
    feat_sin = np.sin(2 * np.pi * p_month/12)
    feat_cos = np.cos(2 * np.pi * p_month/12)
    
    last_3 = current_history.tail(3)
    
    # 1. Predict Frequency (and force to Integer)
    row_freq = pd.DataFrame([{'month_sin': feat_sin, 'month_cos': feat_cos,
                              'Frequency_Lag1': last_3['Frequency'].iloc[-1], 'Frequency_Lag2': last_3['Frequency'].iloc[-2],
                              'Frequency_Lag3': last_3['Frequency'].iloc[-3], 'Frequency_RollMean3': last_3['Frequency'].mean()}])[features_freq]
    pred_freq_raw = model_freq.predict(row_freq)[0]
    pred_freq_int = int(np.round(pred_freq_raw)) # Integer requirement
    
    # 2. Predict Severity
    row_sev = pd.DataFrame([{'month_sin': feat_sin, 'month_cos': feat_cos,
                             'Severity_Lag1': last_3['Severity'].iloc[-1], 'Severity_Lag2': last_3['Severity'].iloc[-2],
                             'Severity_Lag3': last_3['Severity'].iloc[-3], 'Severity_RollMean3': last_3['Severity'].mean()}])[features_sev]
    pred_sev = model_sev.predict(row_sev)[0]
    
    # 3. Predict Total Claim
    row_total = pd.DataFrame([{'month_sin': feat_sin, 'month_cos': feat_cos,
                               'Total_Claim_Lag1': last_3['Total_Claim'].iloc[-1], 'Total_Claim_Lag2': last_3['Total_Claim'].iloc[-2],
                               'Total_Claim_Lag3': last_3['Total_Claim'].iloc[-3], 'Total_Claim_RollMean3': last_3['Total_Claim'].mean()}])[features_total]
    pred_total = model_total.predict(row_total)[0]
    
    # Store Predictions
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_freq_int,
            'Severity': pred_sev,
            'Total_Claim': pred_total 
        })
        
    # Update History for the next step (using the predicted values)
    new_row = pd.DataFrame([{
        'Period': period, 'Year': period.year, 'Month': period.month,
        'Frequency': pred_freq_int, 'Severity': pred_sev, 'Total_Claim': pred_total
    }])
    current_history = pd.concat([current_history, new_row], ignore_index=True)

# ==========================================
# STEP 6: EXPORT
# ==========================================
results_df = pd.DataFrame(predictions)
formatted_data = []

for index, row in results_df.iterrows():
    m_id = row['id_month']
    # Output integer explicitly for Frequency
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': int(row['Frequency'])})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_master_actuarial.csv', index=False)

print("\n--- FINAL FORECAST ---")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nSaved successfully to 'submission_master_actuarial.csv'")

1. Processing Raw Data...
2. Aggregating Monthly Data...
3. Engineering Time-Series Features...
4. Training Actuarial LightGBM Models...
5. Generating Forecast (Aug - Dec 2025)...

--- FINAL FORECAST ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        231  5.477054e+07  1.191360e+10
1      2025-09        237  5.253271e+07  1.141860e+10
2      2025-10        257  5.040581e+07  1.571792e+10
3      2025-11        240  4.992855e+07  1.284961e+10
4      2025-12        237  5.477014e+07  1.124217e+10

Saved successfully to 'submission_master_actuarial.csv'


Hyper-Tuned LightGBM

In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. AGGREGATE RAW DATA
# ==========================================
print("1. Processing Raw Data...")

# Convert dates and filter PAID claims
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims['Period'] = valid_claims['Tanggal Pasien Masuk RS'].dt.to_period('M')

# Aggregate perfectly to match the historical monthly pattern
monthly_data = []
for period, group in valid_claims.groupby('Period'):
    monthly_data.append({
        'Period': period,
        'Year': period.year,
        'Month': period.month,
        'Frequency': len(group), # Absolute Count
        'Severity': group['Nominal Klaim Yang Disetujui'].mean()
    })

df = pd.DataFrame(monthly_data).sort_values('Period').reset_index(drop=True)

# ==========================================
# 2. LEAN FEATURE ENGINEERING
# ==========================================
print("2. Engineering Features...")

# Trend and Seasonality
df['Time_Index'] = np.arange(len(df))
df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)

# Rolling history (Lags 1, 2, 3 and Mean 3)
for col in ['Frequency', 'Severity']:
    df[f'{col}_Lag1'] = df[col].shift(1)
    df[f'{col}_Lag2'] = df[col].shift(2)
    df[f'{col}_Lag3'] = df[col].shift(3)
    df[f'{col}_RollMean3'] = df[col].rolling(window=3).mean()

train_df = df.dropna().copy()
features = ['Time_Index', 'month_sin', 'month_cos', 
            'Frequency_Lag1', 'Frequency_Lag2', 'Frequency_Lag3', 'Frequency_RollMean3',
            'Severity_Lag1', 'Severity_Lag2', 'Severity_Lag3', 'Severity_RollMean3']

# ==========================================
# 3. TRAIN LIGHTGBM (Optimized for Small Data)
# ==========================================
print("3. Training LightGBM Models...")

# Secret Sauce: Restricting the tree size to prevent 10% overfitting gaps
lgbm_params = {
    'objective': 'mape',       # Directly fight Percentage Error
    'metric': 'mape',
    'boosting_type': 'gbdt',
    'n_estimators': 150,       # Lowered to stop memorization
    'learning_rate': 0.05,
    'num_leaves': 7,           # Extremely small leaves for tiny datasets
    'min_child_samples': 2,    # Allow splits on small data
    'random_state': 42,
    'verbose': -1
}

# Train Frequency (Log transformed to stabilize)
model_freq = lgb.LGBMRegressor(**lgbm_params)
model_freq.fit(train_df[features], np.log1p(train_df['Frequency']))

# Train Severity
model_sev = lgb.LGBMRegressor(**lgbm_params)
model_sev.fit(train_df[features], np.log1p(train_df['Severity']))

# ==========================================
# 4. RECURSIVE FORECASTING
# ==========================================
print("4. Forecasting Aug - Dec 2025...")

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=df['Period'].max() + 1, end=target_end, freq='M')

current_history = df.copy()
next_time_index = df['Time_Index'].max() + 1
predictions = []

for period in forecast_range:
    p_month = period.month
    last_3 = current_history.tail(3)
    
    # Build the exact feature row
    input_row = pd.DataFrame([{
        'Time_Index': next_time_index,
        'month_sin': np.sin(2 * np.pi * p_month/12),
        'month_cos': np.cos(2 * np.pi * p_month/12),
        'Frequency_Lag1': last_3['Frequency'].iloc[-1],
        'Frequency_Lag2': last_3['Frequency'].iloc[-2],
        'Frequency_Lag3': last_3['Frequency'].iloc[-3],
        'Frequency_RollMean3': last_3['Frequency'].mean(),
        'Severity_Lag1': last_3['Severity'].iloc[-1],
        'Severity_Lag2': last_3['Severity'].iloc[-2],
        'Severity_Lag3': last_3['Severity'].iloc[-3],
        'Severity_RollMean3': last_3['Severity'].mean()
    }])[features]
    
    # Predict and Reverse Log
    pred_freq_float = np.expm1(model_freq.predict(input_row)[0])
    pred_sev = np.expm1(model_sev.predict(input_row)[0])
    
    # Apply strict rules: Frequency must be an integer
    pred_freq_int = int(np.round(pred_freq_float))
    
    # Total equals integer Count * Severity
    pred_total_claim = pred_freq_int * pred_sev
    
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_freq_int,
            'Severity': pred_sev,
            'Total_Claim': pred_total_claim
        })
        
    # Append to history to calculate the next month's lags
    new_row = pd.DataFrame([{
        'Period': period, 'Year': period.year, 'Month': period.month,
        'Time_Index': next_time_index,
        'Frequency': pred_freq_float, # Keep mathematical float for accurate future lags
        'Severity': pred_sev
    }])
    current_history = pd.concat([current_history, new_row], ignore_index=True)
    next_time_index += 1

# ==========================================
# 5. FORMAT & EXPORT
# ==========================================
results_df = pd.DataFrame(predictions)
formatted_data = []

for index, row in results_df.iterrows():
    m_id = row['id_month']
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_lgbm_hypertuned.csv', index=False)

print("\n--- FINAL FORECAST ---")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_lgbm_hypertuned.csv' saved!")

1. Processing Raw Data...
2. Engineering Features...
3. Training LightGBM Models...
4. Forecasting Aug - Dec 2025...

--- FINAL FORECAST ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        263  4.891353e+07  1.286426e+10
1      2025-09        282  5.207657e+07  1.468559e+10
2      2025-10        260  5.637039e+07  1.465630e+10
3      2025-11        254  5.523338e+07  1.402928e+10
4      2025-12        227  5.519739e+07  1.252981e+10

File 'submission_lgbm_hypertuned.csv' saved!


LightGBM (Aggregate Per Week)

In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

# Convert dates
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])

# Filter PAID claims
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()

# Set index to Date for easy resampling
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Resample to Weekly (W-SUN means weeks end on Sunday)
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

# Rename the date column for clarity (This is the End Date of the week)
weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# Calculate Weekly Severity (Handling weeks with 0 claims)
weekly_df['Severity'] = np.where(weekly_df['Frequency'] > 0, 
                                 weekly_df['Total_Claim'] / weekly_df['Frequency'], 
                                 0)

# ==========================================
# 2. WEEKLY FEATURE ENGINEERING
# ==========================================
print("2. Engineering Weekly Features (~80 rows!)...")

# Extract Week of Year for Seasonality (1-52)
weekly_df['Week_Num'] = weekly_df['Week_End_Date'].dt.isocalendar().week.astype(int)
weekly_df['week_sin'] = np.sin(2 * np.pi * weekly_df['Week_Num']/52.0)
weekly_df['week_cos'] = np.cos(2 * np.pi * weekly_df['Week_Num']/52.0)
weekly_df['Time_Index'] = np.arange(len(weekly_df))

# Lags for the last 4 weeks (equivalent to 1 month of history)
targets = ['Frequency', 'Severity']
for col in targets:
    weekly_df[f'{col}_Lag1'] = weekly_df[col].shift(1)
    weekly_df[f'{col}_Lag2'] = weekly_df[col].shift(2)
    weekly_df[f'{col}_Lag3'] = weekly_df[col].shift(3)
    weekly_df[f'{col}_Lag4'] = weekly_df[col].shift(4)
    weekly_df[f'{col}_RollMean4'] = weekly_df[col].rolling(window=4).mean()

train_df = weekly_df.dropna().copy()
features = ['Time_Index', 'week_sin', 'week_cos', 
            'Frequency_Lag1', 'Frequency_Lag2', 'Frequency_Lag3', 'Frequency_Lag4', 'Frequency_RollMean4',
            'Severity_Lag1', 'Severity_Lag2', 'Severity_Lag3', 'Severity_Lag4', 'Severity_RollMean4']

# ==========================================
# 3. TRAIN LIGHTGBM
# ==========================================
print("3. Training LightGBM on Weekly Data...")

# Because we have ~80 rows, we can use slightly deeper trees now!
lgbm_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 250,
    'learning_rate': 0.05,
    'num_leaves': 15,          # Increased from previous attempts since we have more data
    'min_child_samples': 3,
    'random_state': 42,
    'verbose': -1
}

model_freq = lgb.LGBMRegressor(**lgbm_params)
model_freq.fit(train_df[features], np.log1p(train_df['Frequency']))

model_sev = lgb.LGBMRegressor(**lgbm_params)
model_sev.fit(train_df[features], np.log1p(train_df['Severity']))

# ==========================================
# 4. RECURSIVE FORECASTING (WEEK BY WEEK)
# ==========================================
print("4. Forecasting Future Weeks...")

# We need to forecast until the end of December 2025
last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), 
                               freq='W-SUN')

current_history = weekly_df.copy()
next_time_index = weekly_df['Time_Index'].max() + 1
weekly_predictions = []

for week_end in forecast_weeks:
    week_num = week_end.isocalendar().week
    feat_sin = np.sin(2 * np.pi * week_num/52.0)
    feat_cos = np.cos(2 * np.pi * week_num/52.0)
    
    last_4 = current_history.tail(4)
    
    # Build Feature Row
    input_row = pd.DataFrame([{
        'Time_Index': next_time_index,
        'week_sin': feat_sin, 'week_cos': feat_cos,
        'Frequency_Lag1': last_4['Frequency'].iloc[-1], 'Frequency_Lag2': last_4['Frequency'].iloc[-2],
        'Frequency_Lag3': last_4['Frequency'].iloc[-3], 'Frequency_Lag4': last_4['Frequency'].iloc[-4],
        'Frequency_RollMean4': last_4['Frequency'].mean(),
        'Severity_Lag1': last_4['Severity'].iloc[-1], 'Severity_Lag2': last_4['Severity'].iloc[-2],
        'Severity_Lag3': last_4['Severity'].iloc[-3], 'Severity_Lag4': last_4['Severity'].iloc[-4],
        'Severity_RollMean4': last_4['Severity'].mean()
    }])[features]
    
    # Predict (Convert back from Log)
    pred_freq = np.expm1(model_freq.predict(input_row)[0])
    pred_sev = np.expm1(model_sev.predict(input_row)[0])
    
    # Safety caps
    pred_freq = max(0, pred_freq)
    pred_sev = max(0, pred_sev)
    pred_total = pred_freq * pred_sev
    
    # Store the Weekly Prediction
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })
    
    # Update history for next loop
    new_row = pd.DataFrame([{
        'Week_End_Date': week_end, 'Time_Index': next_time_index,
        'Frequency': pred_freq, 'Severity': pred_sev, 'Total_Claim': pred_total
    }])
    current_history = pd.concat([current_history, new_row], ignore_index=True)
    next_time_index += 1

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []

# Chop every predicted week into 7 exact days
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    
    for d in days:
        daily_records.append({
            'Date': d,
            'Daily_Freq': daily_freq,
            'Daily_Total': daily_total
        })

daily_df = pd.DataFrame(daily_records)

# Extract Year-Month for precise grouping
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

# Group back up to Exact Calendar Months
monthly_forecast = daily_df.groupby('Month_Period').agg(
    Frequency=('Daily_Freq', 'sum'),
    Total_Claim=('Daily_Total', 'sum')
).reset_index()

# Filter ONLY for Target Window (Aug 2025 - Dec 2025)
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

# Apply the strict mathematical rules requested
final_forecast['Frequency'] = np.round(final_forecast['Frequency']).astype(int)

# Total Claim = Frequency * Severity -> therefore Severity = Total_Claim / Integer Frequency
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. FORMAT AND EXPORT
# ==========================================
formatted_data = []

for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_weekly_lgbm.csv', index=False)

print("\n--- FINAL FORECAST (Weekly Aggregation) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_weekly_lgbm.csv' saved! Go submit this!")

1. Processing Raw Data to Weekly Level...
2. Engineering Weekly Features (~80 rows!)...
3. Training LightGBM on Weekly Data...
4. Forecasting Future Weeks...
5. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (Weekly Aggregation) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        214  4.626528e+07  9.900769e+09
1      2025-09        217  4.691034e+07  1.017954e+10
2      2025-10        233  4.996611e+07  1.164210e+10
3      2025-11        226  4.920760e+07  1.112092e+10
4      2025-12        223  4.542997e+07  1.013088e+10

File 'submission_weekly_lgbm.csv' saved! Go submit this!


Weekly Split

In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. HELPER FUNCTION: PREPARE WEEKLY DATA
# ==========================================
def create_weekly_pipeline(df_subset, prefix):
    # Resample to Weekly
    weekly = df_subset.resample('W-SUN').agg(
        Frequency=('Nomor Polis', 'count'),
        Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
    ).reset_index()
    
    weekly.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)
    
    # Severity (Handle weeks with 0 claims safely)
    weekly['Severity'] = np.where(weekly['Frequency'] > 0, 
                                  weekly['Total_Claim'] / weekly['Frequency'], 0)
    
    # Features
    weekly['Week_Num'] = weekly['Week_End_Date'].dt.isocalendar().week.astype(int)
    weekly['week_sin'] = np.sin(2 * np.pi * weekly['Week_Num']/52.0)
    weekly['week_cos'] = np.cos(2 * np.pi * weekly['Week_Num']/52.0)
    weekly['Time_Index'] = np.arange(len(weekly))
    
    for col in ['Frequency', 'Severity']:
        weekly[f'{col}_Lag1'] = weekly[col].shift(1)
        weekly[f'{col}_Lag2'] = weekly[col].shift(2)
        weekly[f'{col}_Lag3'] = weekly[col].shift(3)
        weekly[f'{col}_Lag4'] = weekly[col].shift(4)
        weekly[f'{col}_RollMean4'] = weekly[col].rolling(window=4).mean()
        
    return weekly

# ==========================================
# 2. PROCESS & SPLIT RAW DATA
# ==========================================
print("1. Splitting Data into IP and OP Pipelines...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Split the datasets
claims_ip = valid_claims[valid_claims['Inpatient/Outpatient'] == 'IP'].copy()
claims_op = valid_claims[valid_claims['Inpatient/Outpatient'] == 'OP'].copy()

# Generate Weekly Dataframes
weekly_ip = create_weekly_pipeline(claims_ip, 'IP')
weekly_op = create_weekly_pipeline(claims_op, 'OP')

# ==========================================
# 3. TRAIN INDEPENDENT MODELS
# ==========================================
print("2. Training Separate LightGBM Models for IP and OP...")

features = ['Time_Index', 'week_sin', 'week_cos', 
            'Frequency_Lag1', 'Frequency_Lag2', 'Frequency_Lag3', 'Frequency_Lag4', 'Frequency_RollMean4',
            'Severity_Lag1', 'Severity_Lag2', 'Severity_Lag3', 'Severity_Lag4', 'Severity_RollMean4']

lgbm_params = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'n_estimators': 250, 'learning_rate': 0.05, 'num_leaves': 12,
    'min_child_samples': 3, 'random_state': 42, 'verbose': -1
}

# --- Train IP Models ---
train_ip = weekly_ip.dropna()
model_freq_ip = lgb.LGBMRegressor(**lgbm_params).fit(train_ip[features], np.log1p(train_ip['Frequency']))
model_sev_ip = lgb.LGBMRegressor(**lgbm_params).fit(train_ip[features], np.log1p(train_ip['Severity']))

# --- Train OP Models ---
train_op = weekly_op.dropna()
model_freq_op = lgb.LGBMRegressor(**lgbm_params).fit(train_op[features], np.log1p(train_op['Frequency']))
model_sev_op = lgb.LGBMRegressor(**lgbm_params).fit(train_op[features], np.log1p(train_op['Severity']))

# ==========================================
# 4. RECURSIVE FORECASTING FUNCTION
# ==========================================
print("3. Forecasting Future Weeks...")

def forecast_pipeline(weekly_df, model_freq, model_sev):
    last_hist_date = weekly_df['Week_End_Date'].max()
    target_end_date = pd.to_datetime('2025-12-31')
    forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                                   end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')
    
    current_history = weekly_df.copy()
    next_time_index = weekly_df['Time_Index'].max() + 1
    weekly_predictions = []
    
    for week_end in forecast_weeks:
        week_num = week_end.isocalendar().week
        last_4 = current_history.tail(4)
        
        input_row = pd.DataFrame([{
            'Time_Index': next_time_index,
            'week_sin': np.sin(2 * np.pi * week_num/52.0), 'week_cos': np.cos(2 * np.pi * week_num/52.0),
            'Frequency_Lag1': last_4['Frequency'].iloc[-1], 'Frequency_Lag2': last_4['Frequency'].iloc[-2],
            'Frequency_Lag3': last_4['Frequency'].iloc[-3], 'Frequency_Lag4': last_4['Frequency'].iloc[-4],
            'Frequency_RollMean4': last_4['Frequency'].mean(),
            'Severity_Lag1': last_4['Severity'].iloc[-1], 'Severity_Lag2': last_4['Severity'].iloc[-2],
            'Severity_Lag3': last_4['Severity'].iloc[-3], 'Severity_Lag4': last_4['Severity'].iloc[-4],
            'Severity_RollMean4': last_4['Severity'].mean()
        }])[features]
        
        pred_freq = max(0, np.expm1(model_freq.predict(input_row)[0]))
        pred_sev = max(0, np.expm1(model_sev.predict(input_row)[0]))
        pred_total = pred_freq * pred_sev
        
        weekly_predictions.append({
            'Week_Start_Date': week_end - pd.Timedelta(days=6),
            'Week_End_Date': week_end,
            'Frequency': pred_freq,
            'Total_Claim': pred_total
        })
        
        new_row = pd.DataFrame([{'Week_End_Date': week_end, 'Time_Index': next_time_index,
                                 'Frequency': pred_freq, 'Severity': pred_sev, 'Total_Claim': pred_total}])
        current_history = pd.concat([current_history, new_row], ignore_index=True)
        next_time_index += 1
        
    return weekly_predictions

pred_weekly_ip = forecast_pipeline(weekly_ip, model_freq_ip, model_sev_ip)
pred_weekly_op = forecast_pipeline(weekly_op, model_freq_op, model_sev_op)

# ==========================================
# 5. DAILY APPORTIONMENT & RE-ASSEMBLY
# ==========================================
print("4. Apportioning to Daily and Assembling Final Calendar Months...")

def apportion_to_monthly(weekly_preds, suffix):
    daily_records = []
    for row in weekly_preds:
        days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
        daily_freq = row['Frequency'] / 7.0
        daily_total = row['Total_Claim'] / 7.0
        for d in days:
            daily_records.append({'Date': d, f'Daily_Freq_{suffix}': daily_freq, f'Daily_Total_{suffix}': daily_total})
            
    daily_df = pd.DataFrame(daily_records)
    daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')
    
    # THE FIX: Explicitly select ONLY the numeric columns before summing
    cols_to_sum = [f'Daily_Freq_{suffix}', f'Daily_Total_{suffix}']
    return daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

monthly_ip = apportion_to_monthly(pred_weekly_ip, 'IP')
monthly_op = apportion_to_monthly(pred_weekly_op, 'OP')

# Merge IP and OP back together
final_monthly = pd.merge(monthly_ip, monthly_op, on='Month_Period')

# Calculate the Grand Totals
final_monthly['Total_Frequency'] = final_monthly['Daily_Freq_IP'] + final_monthly['Daily_Freq_OP']
final_monthly['Total_Claim_All'] = final_monthly['Daily_Total_IP'] + final_monthly['Daily_Total_OP']

# Filter Target Window
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = final_monthly[(final_monthly['Month_Period'] >= target_start) & 
                               (final_monthly['Month_Period'] <= target_end)].copy()

# Apply Strict Leaderboard Rules
final_forecast['Frequency'] = np.round(final_forecast['Total_Frequency']).astype(int)
final_forecast['Severity'] = final_forecast['Total_Claim_All'] / final_forecast['Frequency']
final_forecast['Total_Claim'] = final_forecast['Total_Claim_All']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_weekly_split.csv', index=False)

print("\n--- FINAL FORECAST (Weekly IP/OP Split) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_weekly_split.csv' saved!")

1. Splitting Data into IP and OP Pipelines...
2. Training Separate LightGBM Models for IP and OP...
3. Forecasting Future Weeks...
4. Apportioning to Daily and Assembling Final Calendar Months...

--- FINAL FORECAST (Weekly IP/OP Split) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        187  3.971136e+07  7.426024e+09
1      2025-09        199  4.183472e+07  8.325109e+09
2      2025-10        211  4.176483e+07  8.812380e+09
3      2025-11        203  4.072161e+07  8.266486e+09
4      2025-12        200  3.747824e+07  7.495649e+09

File 'submission_weekly_split.csv' saved!


LightGBM 5 Days Aggregate

In [11]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO 5-DAY AGGREGATION
# ==========================================
print("1. Processing Raw Data to 5-Day Blocks...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Resample into strict 5-Day blocks. 
# By default, Pandas labels the 'start' of the period.
df_5d = valid_claims.resample('5D').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

df_5d.rename(columns={'Tanggal Pasien Masuk RS': 'Period_Start_Date'}, inplace=True)

# Calculate the exact End Date of the 5-day block (Start Date + 4 days)
df_5d['Period_End_Date'] = df_5d['Period_Start_Date'] + pd.Timedelta(days=4)

# Calculate Severity for historical reference
df_5d['Severity'] = np.where(df_5d['Frequency'] > 0, 
                             df_5d['Total_Claim'] / df_5d['Frequency'], 0)

# ==========================================
# 2. 5-DAY FEATURE ENGINEERING
# ==========================================
print(f"2. Engineering Features for {len(df_5d)} rows of 5-Day data...")

# Because 5 days drifts across the calendar, we give the model the Month context
df_5d['Month'] = df_5d['Period_End_Date'].dt.month
df_5d['month_sin'] = np.sin(2 * np.pi * df_5d['Month']/12.0)
df_5d['month_cos'] = np.cos(2 * np.pi * df_5d['Month']/12.0)
df_5d['Time_Index'] = np.arange(len(df_5d))

# Lags (Looking back 1, 2, 3, 4, 5, and 6 blocks = roughly 1 month of history)
targets = ['Frequency', 'Severity']
for col in targets:
    for i in range(1, 7):
        df_5d[f'{col}_Lag{i}'] = df_5d[col].shift(i)
    df_5d[f'{col}_RollMean6'] = df_5d[col].rolling(window=6).mean()

train_df = df_5d.dropna().copy()

features = ['Time_Index', 'month_sin', 'month_cos'] + \
           [f'Frequency_Lag{i}' for i in range(1, 7)] + ['Frequency_RollMean6'] + \
           [f'Severity_Lag{i}' for i in range(1, 7)] + ['Severity_RollMean6']

# ==========================================
# 3. TRAIN LIGHTGBM
# ==========================================
print("3. Training LightGBM on 5-Day Intervals...")

lgbm_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 250,
    'learning_rate': 0.05,
    'num_leaves': 15,          
    'min_child_samples': 3,
    'random_state': 42,
    'verbose': -1
}

model_freq = lgb.LGBMRegressor(**lgbm_params)
model_freq.fit(train_df[features], np.log1p(train_df['Frequency']))

model_sev = lgb.LGBMRegressor(**lgbm_params)
model_sev.fit(train_df[features], np.log1p(train_df['Severity']))

# ==========================================
# 4. RECURSIVE FORECASTING (BLOCK BY BLOCK)
# ==========================================
print("4. Forecasting Future 5-Day Blocks...")

last_start_date = df_5d['Period_Start_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')

# Generate future 5-day periods
forecast_periods = pd.date_range(start=last_start_date + pd.Timedelta(days=5), 
                                 end=target_end_date + pd.Timedelta(days=5), 
                                 freq='5D')

current_history = df_5d.copy()
next_time_index = df_5d['Time_Index'].max() + 1
period_predictions = []

for p_start in forecast_periods:
    p_end = p_start + pd.Timedelta(days=4)
    p_month = p_end.month
    
    last_6 = current_history.tail(6)
    
    input_dict = {
        'Time_Index': next_time_index,
        'month_sin': np.sin(2 * np.pi * p_month/12.0),
        'month_cos': np.cos(2 * np.pi * p_month/12.0)
    }
    
    for i in range(1, 7):
        input_dict[f'Frequency_Lag{i}'] = last_6['Frequency'].iloc[-i]
        input_dict[f'Severity_Lag{i}'] = last_6['Severity'].iloc[-i]
        
    input_dict['Frequency_RollMean6'] = last_6['Frequency'].mean()
    input_dict['Severity_RollMean6'] = last_6['Severity'].mean()
    
    input_row = pd.DataFrame([input_dict])[features]
    
    pred_freq = np.expm1(model_freq.predict(input_row)[0])
    pred_sev = np.expm1(model_sev.predict(input_row)[0])
    
    pred_freq = max(0, pred_freq)
    pred_sev = max(0, pred_sev)
    pred_total = pred_freq * pred_sev
    
    period_predictions.append({
        'Period_Start_Date': p_start,
        'Period_End_Date': p_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })
    
    new_row = pd.DataFrame([{
        'Period_Start_Date': p_start, 'Period_End_Date': p_end, 'Time_Index': next_time_index,
        'Frequency': pred_freq, 'Severity': pred_sev, 'Total_Claim': pred_total
    }])
    current_history = pd.concat([current_history, new_row], ignore_index=True)
    next_time_index += 1

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []

# Chop every predicted 5-day block into 5 exact days
for row in period_predictions:
    days = pd.date_range(start=row['Period_Start_Date'], end=row['Period_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 5.0    # DIVIDED BY 5 NOW!
    daily_total = row['Total_Claim'] / 5.0 # DIVIDED BY 5 NOW!
    
    for d in days:
        daily_records.append({
            'Date': d,
            'Daily_Freq': daily_freq,
            'Daily_Total': daily_total
        })

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

# Explicitly select numeric columns to avoid the pandas date-summing crash
cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

# Filter ONLY for Target Window
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

# The Mathematical Guarantee Rule
final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_5day_lgbm.csv', index=False)

print("\n--- FINAL FORECAST (5-Day Aggregation) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_5day_lgbm.csv' saved!")

1. Processing Raw Data to 5-Day Blocks...
2. Engineering Features for 116 rows of 5-Day data...
3. Training LightGBM on 5-Day Intervals...
4. Forecasting Future 5-Day Blocks...
5. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (5-Day Aggregation) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        220  4.742200e+07  1.043284e+10
1      2025-09        201  4.641158e+07  9.328728e+09
2      2025-10        189  5.115159e+07  9.667650e+09
3      2025-11        202  5.085614e+07  1.027294e+10
4      2025-12        194  5.074567e+07  9.844661e+09

File 'submission_5day_lgbm.csv' saved!


DLinear

In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Resample to Weekly
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# ==========================================
# 2. DLINEAR DECOMPOSITION & FEATURE ENGINEERING
# ==========================================
print("2. Performing DLinear Decomposition...")

window_size = 4 # 4-week moving average for Trend

targets = ['Frequency', 'Total_Claim']

for col in targets:
    # 1. Extract Trend (Moving Average)
    weekly_df[f'{col}_Trend'] = weekly_df[col].rolling(window=window_size).mean()
    
    # 2. Extract Remainder (Raw - Trend)
    weekly_df[f'{col}_Remain'] = weekly_df[col] - weekly_df[f'{col}_Trend']
    
    # 3. Create Lags for the Linear Layers (Lookback = 4 weeks)
    for lag in range(1, 5):
        weekly_df[f'{col}_Trend_Lag{lag}'] = weekly_df[f'{col}_Trend'].shift(lag)
        weekly_df[f'{col}_Remain_Lag{lag}'] = weekly_df[f'{col}_Remain'].shift(lag)

train_df = weekly_df.dropna().copy()

# ==========================================
# 3. TRAIN DLINEAR LINEAR LAYERS
# ==========================================
print("3. Training Dual Linear Layers...")

# We use Ridge (Regularized Linear Regression) to prevent wild coefficient spikes
model_params = {'alpha': 1.0}

# --- FREQUENCY MODELS ---
features_freq_trend = [f'Frequency_Trend_Lag{i}' for i in range(1, 5)]
model_freq_trend = Ridge(**model_params)
model_freq_trend.fit(train_df[features_freq_trend], train_df['Frequency_Trend'])

features_freq_remain = [f'Frequency_Remain_Lag{i}' for i in range(1, 5)]
model_freq_remain = Ridge(**model_params)
model_freq_remain.fit(train_df[features_freq_remain], train_df['Frequency_Remain'])

# --- TOTAL CLAIM MODELS ---
features_tot_trend = [f'Total_Claim_Trend_Lag{i}' for i in range(1, 5)]
model_tot_trend = Ridge(**model_params)
model_tot_trend.fit(train_df[features_tot_trend], train_df['Total_Claim_Trend'])

features_tot_remain = [f'Total_Claim_Remain_Lag{i}' for i in range(1, 5)]
model_tot_remain = Ridge(**model_params)
model_tot_remain.fit(train_df[features_tot_remain], train_df['Total_Claim_Remain'])

# ==========================================
# 4. RECURSIVE WEEKLY FORECASTING
# ==========================================
print("4. Forecasting Future Weeks recursively...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

current_history = weekly_df.copy()
weekly_predictions = []

for week_end in forecast_weeks:
    # Get the last 4 known rows to build our lag features
    last_4 = current_history.tail(4)
    
    # --- FREQUENCY PREDICTION ---
    row_freq_trend = pd.DataFrame([{f'Frequency_Trend_Lag{i}': last_4['Frequency_Trend'].iloc[-i] for i in range(1, 5)}])
    row_freq_remain = pd.DataFrame([{f'Frequency_Remain_Lag{i}': last_4['Frequency_Remain'].iloc[-i] for i in range(1, 5)}])
    
    pred_f_trend = model_freq_trend.predict(row_freq_trend)[0]
    pred_f_remain = model_freq_remain.predict(row_freq_remain)[0]
    
    # DLinear Recombination: Output = Trend + Remainder
    pred_freq = pred_f_trend + pred_f_remain
    pred_freq = max(0, pred_freq) # Safety cap
    
    # --- TOTAL CLAIM PREDICTION ---
    row_tot_trend = pd.DataFrame([{f'Total_Claim_Trend_Lag{i}': last_4['Total_Claim_Trend'].iloc[-i] for i in range(1, 5)}])
    row_tot_remain = pd.DataFrame([{f'Total_Claim_Remain_Lag{i}': last_4['Total_Claim_Remain'].iloc[-i] for i in range(1, 5)}])
    
    pred_t_trend = model_tot_trend.predict(row_tot_trend)[0]
    pred_t_remain = model_tot_remain.predict(row_tot_remain)[0]
    
    pred_total = pred_t_trend + pred_t_remain
    pred_total = max(0, pred_total)
    
    # Store Prediction
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })
    
    # Update history for the next loop so the Moving Average updates correctly
    new_row_data = {'Week_End_Date': week_end, 'Frequency': pred_freq, 'Total_Claim': pred_total}
    
    # Calculate the new Trend and Remainder based on this new prediction
    hist_plus_new = pd.concat([current_history, pd.DataFrame([new_row_data])], ignore_index=True)
    
    # Update Decomposition mathematically
    for col in targets:
        hist_plus_new[f'{col}_Trend'] = hist_plus_new[col].rolling(window=window_size).mean()
        hist_plus_new[f'{col}_Remain'] = hist_plus_new[col] - hist_plus_new[f'{col}_Trend']
        
    current_history = hist_plus_new.copy()

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

# Roll up explicitly selecting numeric columns to prevent datetime sum errors
cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

# Filter for Target Window (Aug 2025 - Dec 2025)
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

# Mathematical Alignments
final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
# Derived Severity to prevent compounding evaluation errors
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_dlinear.csv', index=False)

print("\n--- FINAL FORECAST (DLinear Architecture) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_dlinear.csv' saved! Time to drop it in the leaderboard!")

1. Processing Raw Data to Weekly Level...
2. Performing DLinear Decomposition...
3. Training Dual Linear Layers...
4. Forecasting Future Weeks recursively...
5. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (DLinear Architecture) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        220  5.054340e+07  1.111955e+10
1      2025-09        237  5.233406e+07  1.240317e+10
2      2025-10        247  5.299879e+07  1.309070e+10
3      2025-11        239  5.341499e+07  1.276618e+10
4      2025-12        248  5.334536e+07  1.322965e+10

File 'submission_dlinear.csv' saved! Time to drop it in the leaderboard!


Chronos (DL)

In [10]:
import pandas as pd
import numpy as np
import torch
from chronos import ChronosPipeline
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

# Assuming your data is loaded as data_claim
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Resample to Weekly (The trick that got you 7%!)
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# Calculate Severity for historical context
weekly_df['Severity'] = np.where(weekly_df['Frequency'] > 0, 
                                 weekly_df['Total_Claim'] / weekly_df['Frequency'], 0)

# ==========================================
# 2. LOAD CHRONOS FOUNDATION MODEL
# ==========================================
print("2. Downloading/Loading Chronos Foundation Model...")

# We use the 'small' version which easily runs on a standard laptop CPU/GPU.
# If you have a powerful GPU, you can change 'small' to 'base' or 'large'
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="auto", # Automatically uses GPU if you have one, otherwise CPU
    torch_dtype=torch.float32
)

# ==========================================
# 3. ZERO-SHOT FORECASTING
# ==========================================
print("3. Asking Chronos to predict the future...")

# We need to forecast until the end of Dec 2025. 
# Let's forecast 25 weeks into the future to be safe and cover the whole window.
forecast_length = 25 

# Convert your historical weekly columns into simple PyTorch tensors
context_freq = torch.tensor(weekly_df['Frequency'].values, dtype=torch.float32)
context_sev = torch.tensor(weekly_df['Severity'].values, dtype=torch.float32)

# Chronos predicts 20 possible futures (samples) for every single week.
# This gives us a statistical distribution.
forecast_freq_dist = pipeline.predict(context_freq, prediction_length=forecast_length)
forecast_sev_dist = pipeline.predict(context_sev, prediction_length=forecast_length)

# We want the 50th percentile (the Median/most likely prediction)
pred_freq_tensor = torch.quantile(forecast_freq_dist[0], 0.5, dim=0)
pred_sev_tensor = torch.quantile(forecast_sev_dist[0], 0.5, dim=0)

# Convert back to standard arrays by moving to CPU and making it a list
pred_freq_array = pred_freq_tensor.cpu().tolist()
pred_sev_array = pred_sev_tensor.cpu().tolist()

# ==========================================
# 4. MAP PREDICTIONS TO DATES
# ==========================================
print("4. Mapping predictions to future calendar weeks...")

last_hist_date = weekly_df['Week_End_Date'].max()
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               periods=forecast_length, freq='W-SUN')

weekly_predictions = []

for i, week_end in enumerate(forecast_weeks):
    # Get the predicted values
    pred_freq = max(0, pred_freq_array[i]) # Ensure no negative predictions
    pred_sev = max(0, pred_sev_array[i])
    pred_total = pred_freq * pred_sev
    
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

# Roll up explicitly selecting numeric columns
cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

# Filter for Target Window (Aug 2025 - Dec 2025)
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

# The Mathematical Alignments (To prevent leaderboard compounding errors)
final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_chronos.csv', index=False)

print("\n--- FINAL FORECAST (Chronos-T5 Zero-Shot) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_chronos.csv' saved!")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\ipykernel\kernelapp.py", line 739, in st

1. Processing Raw Data to Weekly Level...
2. Downloading/Loading Chronos Foundation Model...


`torch_dtype` is deprecated! Use `dtype` instead!


3. Asking Chronos to predict the future...
4. Mapping predictions to future calendar weeks...
5. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (Chronos-T5 Zero-Shot) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        201  5.101191e+07  1.025339e+10
1      2025-09        223  4.903403e+07  1.093459e+10
2      2025-10        226  4.885168e+07  1.104048e+10
3      2025-11        231  4.811009e+07  1.111343e+10
4      2025-12        242  4.477731e+07  1.083611e+10

File 'submission_chronos.csv' saved!


LightGBM Weekly + ARIMA Stacking

In [12]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

weekly_df['Severity'] = np.where(weekly_df['Frequency'] > 0, 
                                 weekly_df['Total_Claim'] / weekly_df['Frequency'], 0)

# ==========================================
# 2. TRAIN ARIMA TO GENERATE "BASELINE" FEATURES
# ==========================================
print("2. Training ARIMA models to generate the trend baseline...")

# We use a standard ARIMA(1,1,1) order which works great for weekly trend smoothing
model_arima_freq = ARIMA(weekly_df['Frequency'], order=(1, 1, 1)).fit()
model_arima_sev = ARIMA(weekly_df['Severity'], order=(1, 1, 1)).fit()

# Extract the "In-Sample" predictions (What ARIMA thought the past looked like)
weekly_df['ARIMA_Freq_Pred'] = model_arima_freq.predict(start=0, end=len(weekly_df)-1)
weekly_df['ARIMA_Sev_Pred'] = model_arima_sev.predict(start=0, end=len(weekly_df)-1)

# Clean up any weird negative predictions ARIMA might have made early on
weekly_df['ARIMA_Freq_Pred'] = weekly_df['ARIMA_Freq_Pred'].clip(lower=0)
weekly_df['ARIMA_Sev_Pred'] = weekly_df['ARIMA_Sev_Pred'].clip(lower=0)

# ==========================================
# 3. WEEKLY FEATURE ENGINEERING (WITH ARIMA)
# ==========================================
print("3. Engineering Features (Combining Lags + ARIMA)...")

weekly_df['Week_Num'] = weekly_df['Week_End_Date'].dt.isocalendar().week.astype(int)
weekly_df['week_sin'] = np.sin(2 * np.pi * weekly_df['Week_Num']/52.0)
weekly_df['week_cos'] = np.cos(2 * np.pi * weekly_df['Week_Num']/52.0)
weekly_df['Time_Index'] = np.arange(len(weekly_df))

targets = ['Frequency', 'Severity']
for col in targets:
    weekly_df[f'{col}_Lag1'] = weekly_df[col].shift(1)
    weekly_df[f'{col}_Lag2'] = weekly_df[col].shift(2)
    weekly_df[f'{col}_Lag3'] = weekly_df[col].shift(3)
    weekly_df[f'{col}_Lag4'] = weekly_df[col].shift(4)
    weekly_df[f'{col}_RollMean4'] = weekly_df[col].rolling(window=4).mean()

train_df = weekly_df.dropna().copy()

# Notice the new ARIMA features are injected directly into the tree!
features_freq = ['Time_Index', 'week_sin', 'week_cos', 'ARIMA_Freq_Pred',
                 'Frequency_Lag1', 'Frequency_Lag2', 'Frequency_Lag3', 'Frequency_Lag4', 'Frequency_RollMean4']

features_sev = ['Time_Index', 'week_sin', 'week_cos', 'ARIMA_Sev_Pred',
                'Severity_Lag1', 'Severity_Lag2', 'Severity_Lag3', 'Severity_Lag4', 'Severity_RollMean4']

# ==========================================
# 4. TRAIN LIGHTGBM
# ==========================================
print("4. Training LightGBM on Lags + ARIMA Baseline...")

lgbm_params = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'n_estimators': 250, 'learning_rate': 0.05, 'num_leaves': 15,
    'min_child_samples': 3, 'random_state': 42, 'verbose': -1
}

model_lgb_freq = lgb.LGBMRegressor(**lgbm_params)
model_lgb_freq.fit(train_df[features_freq], np.log1p(train_df['Frequency']))

model_lgb_sev = lgb.LGBMRegressor(**lgbm_params)
model_lgb_sev.fit(train_df[features_sev], np.log1p(train_df['Severity']))

# ==========================================
# 5. RECURSIVE FORECASTING
# ==========================================
print("5. Forecasting Future Weeks...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

# Pre-calculate the out-of-sample future ARIMA predictions
steps_to_forecast = len(forecast_weeks)
future_arima_freq = model_arima_freq.forecast(steps=steps_to_forecast).values
future_arima_sev = model_arima_sev.forecast(steps=steps_to_forecast).values

current_history = weekly_df.copy()
next_time_index = weekly_df['Time_Index'].max() + 1
weekly_predictions = []

for i, week_end in enumerate(forecast_weeks):
    week_num = week_end.isocalendar().week
    last_4 = current_history.tail(4)
    
    # Grab the exact ARIMA prediction for this future step
    current_arima_freq = max(0, future_arima_freq[i])
    current_arima_sev = max(0, future_arima_sev[i])
    
    # Build Input Row for Frequency
    input_freq = pd.DataFrame([{
        'Time_Index': next_time_index,
        'week_sin': np.sin(2 * np.pi * week_num/52.0), 'week_cos': np.cos(2 * np.pi * week_num/52.0),
        'ARIMA_Freq_Pred': current_arima_freq,
        'Frequency_Lag1': last_4['Frequency'].iloc[-1], 'Frequency_Lag2': last_4['Frequency'].iloc[-2],
        'Frequency_Lag3': last_4['Frequency'].iloc[-3], 'Frequency_Lag4': last_4['Frequency'].iloc[-4],
        'Frequency_RollMean4': last_4['Frequency'].mean()
    }])[features_freq]
    
    # Build Input Row for Severity
    input_sev = pd.DataFrame([{
        'Time_Index': next_time_index,
        'week_sin': np.sin(2 * np.pi * week_num/52.0), 'week_cos': np.cos(2 * np.pi * week_num/52.0),
        'ARIMA_Sev_Pred': current_arima_sev,
        'Severity_Lag1': last_4['Severity'].iloc[-1], 'Severity_Lag2': last_4['Severity'].iloc[-2],
        'Severity_Lag3': last_4['Severity'].iloc[-3], 'Severity_Lag4': last_4['Severity'].iloc[-4],
        'Severity_RollMean4': last_4['Severity'].mean()
    }])[features_sev]
    
    # LightGBM adjusts the ARIMA prediction based on the lags!
    pred_freq = np.expm1(model_lgb_freq.predict(input_freq)[0])
    pred_sev = np.expm1(model_lgb_sev.predict(input_sev)[0])
    
    pred_freq = max(0, pred_freq)
    pred_sev = max(0, pred_sev)
    pred_total = pred_freq * pred_sev
    
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6), 'Week_End_Date': week_end,
        'Frequency': pred_freq, 'Total_Claim': pred_total
    })
    
    new_row = pd.DataFrame([{'Week_End_Date': week_end, 'Time_Index': next_time_index,
                             'Frequency': pred_freq, 'Severity': pred_sev, 'Total_Claim': pred_total}])
    current_history = pd.concat([current_history, new_row], ignore_index=True)
    next_time_index += 1

# ==========================================
# 6. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("6. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 7. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_lgbm_arima_stack.csv', index=False)

print("\n--- FINAL FORECAST (LGBM + ARIMA Stacking) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_lgbm_arima_stack.csv' saved!")

1. Processing Raw Data to Weekly Level...
2. Training ARIMA models to generate the trend baseline...
3. Engineering Features (Combining Lags + ARIMA)...
4. Training LightGBM on Lags + ARIMA Baseline...
5. Forecasting Future Weeks...
6. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (LGBM + ARIMA Stacking) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        227  4.331576e+07  9.832678e+09
1      2025-09        245  4.332449e+07  1.061450e+10
2      2025-10        267  3.811709e+07  1.017726e+10
3      2025-11        264  4.224458e+07  1.115257e+10
4      2025-12        255  3.657475e+07  9.326562e+09

File 'submission_lgbm_arima_stack.csv' saved!


Auto-SARIMA Weekly

In [14]:
import pandas as pd
import numpy as np
import pmdarima as pm
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Resample to Weekly (The legendary 7% trick)
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

weekly_df['Severity'] = np.where(weekly_df['Frequency'] > 0, 
                                 weekly_df['Total_Claim'] / weekly_df['Frequency'], 0)

# ==========================================
# 2. AUTO-SARIMA TRAINING
# ==========================================
print("2. Running Auto-SARIMA to find the perfect mathematical parameters...")

# m=4 means we are telling the model "Look for a repeating pattern every 4 weeks (Monthly cycle)"
# We don't use m=52 (yearly) because we don't have enough years of data to support it.

print("   -> Tuning Frequency Model...")
model_freq = pm.auto_arima(weekly_df['Frequency'], 
                           seasonal=True, m=4,
                           stepwise=True, suppress_warnings=True, 
                           error_action="ignore", trace=True)

print("   -> Tuning Severity Model...")
model_sev = pm.auto_arima(weekly_df['Severity'], 
                          seasonal=True, m=4,
                          stepwise=True, suppress_warnings=True, 
                          error_action="ignore", trace=True)

print(f"\n[WINNING FREQ MODEL]: {model_freq.summary().tables[0].data[0][1]}")
print(f"[WINNING SEV MODEL]: {model_sev.summary().tables[0].data[0][1]}")

# ==========================================
# 3. FORECASTING FUTURE WEEKS
# ==========================================
print("\n3. Forecasting Future Weeks...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')

forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

steps_to_forecast = len(forecast_weeks)

# Auto-ARIMA forecasts the entire array of future steps instantly
future_freq = model_freq.predict(n_periods=steps_to_forecast).values
future_sev = model_sev.predict(n_periods=steps_to_forecast).values

weekly_predictions = []

for i, week_end in enumerate(forecast_weeks):
    pred_freq = max(0, future_freq[i])
    pred_sev = max(0, future_sev[i])
    pred_total = pred_freq * pred_sev
    
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })

# ==========================================
# 4. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("4. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 5. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_auto_sarima.csv', index=False)

print("\n--- FINAL FORECAST (Auto-SARIMA) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_auto_sarima.csv' saved!")

1. Processing Raw Data to Weekly Level...
2. Running Auto-SARIMA to find the perfect mathematical parameters...
   -> Tuning Frequency Model...
Performing stepwise search to minimize aic
 ARIMA(2,0,2)(1,0,1)[4] intercept   : AIC=660.448, Time=0.38 sec
 ARIMA(0,0,0)(0,0,0)[4] intercept   : AIC=656.550, Time=0.01 sec
 ARIMA(1,0,0)(1,0,0)[4] intercept   : AIC=657.393, Time=0.16 sec
 ARIMA(0,0,1)(0,0,1)[4] intercept   : AIC=657.009, Time=0.07 sec
 ARIMA(0,0,0)(0,0,0)[4]             : AIC=908.965, Time=0.01 sec
 ARIMA(0,0,0)(1,0,0)[4] intercept   : AIC=656.644, Time=0.08 sec
 ARIMA(0,0,0)(0,0,1)[4] intercept   : AIC=656.500, Time=0.04 sec
 ARIMA(0,0,0)(1,0,1)[4] intercept   : AIC=657.585, Time=0.20 sec
 ARIMA(0,0,0)(0,0,2)[4] intercept   : AIC=658.345, Time=0.07 sec
 ARIMA(0,0,0)(1,0,2)[4] intercept   : AIC=660.433, Time=0.29 sec
 ARIMA(1,0,0)(0,0,1)[4] intercept   : AIC=657.257, Time=0.11 sec
 ARIMA(1,0,1)(0,0,1)[4] intercept   : AIC=659.037, Time=0.08 sec
 ARIMA(0,0,0)(0,0,1)[4]          